# 04: Distress label (Phase 4)

**Goal:** define financial distress with a clear rule, attach next year's label to this year's ratios, and report the class balance before modelling.

**Input:** `data/processed/ratios.csv` (Phase 3).

**Outputs:**
- `data/processed/zscores.csv`: Z' components, score, zone and both same-year labels for every company-year
- `data/processed/modelling_table.csv`: features from year t with the labels of year t+1 (feature years 2015 to 2024 labelled; 2025 rows unlabelled, kept for scoring the case companies)
- `data/processed/class_balance.csv`: class counts by feature year

**Primary label (Altman Z', firms without market prices):**
Z' = 0.717 X1 + 0.847 X2 + 3.107 X3 + 0.420 X4 + 0.998 X5, distressed if Z' < 1.23; the grey zone (1.23 to 2.90) counts as healthy. Z' and X1 to X5 are used only to build the label, never as model inputs.

**Robustness label:** net loss, interest coverage < 1.5 and current ratio < 1 in the same year.

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "..")
from src import config
from src import labels as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

ratios = pd.read_csv(config.PROCESSED_DIR / "ratios.csv", parse_dates=["fiscal_year_end"])
print(f"Ratios: {len(ratios):,} company-years, {ratios['cik'].nunique()} companies")

Ratios: 1,028 company-years, 161 companies


## Step 1: Z' components and score

| Term | Formula | Meaning |
| --- | --- | --- |
| X1 | (current assets − current liabilities) / total assets | working capital |
| X2 | retained earnings / total assets | accumulated profitability (and age) |
| X3 | operating income / total assets | operating return on assets |
| X4 | book equity / total liabilities | solvency cushion |
| X5 | revenue / total assets | asset turnover |

In [2]:
z = L.altman_z(ratios)
z["robust_label"] = L.robustness_label(ratios)
zs = pd.concat([ratios[["cik", "name", "label_year", "fiscal_year_end", "low_revenue"]], z], axis=1)

print(f"Z' computed: {zs['z_prime'].notna().sum():,} of {len(zs):,} company-years")
print("Missing component (Z' blank, so no label):", z[["X1", "X2", "X3", "X4", "X5"]].isna().sum().to_dict())
zs[["X1", "X2", "X3", "X4", "X5", "z_prime"]].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

Z' computed: 983 of 1,028 company-years
Missing component (Z' blank, so no label): {'X1': 1, 'X2': 28, 'X3': 19, 'X4': 0, 'X5': 0}


,count,mean,std,min,1%,25%,50%,75%,99%,max
X1,"1,027.00",-2.94,75.45,"-2,302.28",-5.08,0.21,0.35,0.53,0.89,0.96
X2,"1,000.00",-8.81,90.51,"-2,303.28",-75.17,-1.80,-0.13,0.29,1.62,18.40
X3,"1,009.00",15.85,562.12,"-1,581.24",-2.63,-0.13,0.02,0.09,0.35,"17,783.70"
X4,"1,028.00",3.37,7.02,-1.30,-0.88,0.78,1.63,3.93,29.51,99.06
X5,"1,028.00",0.71,2.14,-0.20,0.00,0.43,0.60,0.82,2.00,67.95
z_prime,983.00,43.38,"1,750.69","-8,514.95",-62.87,-0.04,1.55,2.75,15.20,"54,192.80"


## Step 2: Same-year zones across the sector

Share of semiconductor companies in each Z' zone by label year (analysis years 2015 to 2025).

In [3]:
a = zs[zs["label_year"].between(config.LABEL_YEAR_START, config.LABEL_YEAR_END)]
zones = pd.crosstab(a["label_year"], a["zone"])
zones["companies_with_z"] = zones.sum(axis=1)
zones["share_distressed"] = zones["distress"] / zones["companies_with_z"]
zones

zone,distress,grey,safe,companies_with_z,share_distressed
label_year,,,,,
2015,26,38,21,85,0.31
2016,38,20,19,77,0.49
2017,37,24,20,81,0.46
2018,33,27,22,82,0.40
2019,38,27,16,81,0.47
2020,31,35,12,78,0.40
2021,38,30,17,85,0.45
2022,31,34,17,82,0.38
2023,32,32,16,80,0.40


In [4]:
# Distress is concentrated among very small companies (revenue below USD 10 million).
a.groupby("low_revenue")["distressed"].agg(rows="count", distressed="sum", share="mean")

,rows,distressed,share
low_revenue,,,
False,792,292.00,0.37
True,97,85.00,0.88


## Step 3: Robustness label (same year) and agreement with Z'

Three-valued logic: interest = 0 because there is no debt counts as "coverage not weak"; the label is 1 only if all three conditions are known to hold, 0 if any known condition fails, and blank otherwise.

In [5]:
print("Robustness label (analysis years):", a["robust_label"].value_counts(dropna=False).to_dict())
pd.crosstab(a["distressed"], a["robust_label"], rownames=["Z' distressed"], colnames=["robustness label"], margins=True)

Robustness label (analysis years): {0.0: 875, 1.0: 36, nan: 14}


robustness label,0.00,1.00,All
Z' distressed,,,
0.00,512,0,512
1.00,333,36,369
All,845,36,881


## Step 4: Time shift (features at t, label at t+1)

Each feature-year row gets the same company's label from the **immediately following** label year. A missing next year leaves the label blank (the row cannot be used for training or testing), so year t is never paired with t+2.

In [6]:
base = pd.concat([ratios, z[["z_prime", "distressed", "robust_label"]]], axis=1)
shifted = L.shift_to_next_year(base, ["distressed", "robust_label", "z_prime"])

features_years = shifted["label_year"].between(config.LABEL_YEAR_START, config.LABEL_YEAR_END)
mt = shifted[features_years].copy()
mt["has_label"] = mt["distressed_t1"].notna()

print(f"Feature-year rows {config.LABEL_YEAR_START}-{config.LABEL_YEAR_END}: {len(mt):,}")
print(f"  with a t+1 primary label:  {mt['has_label'].sum():,}")
print(f"  without (no next-year row or next-year Z' blank): {(~mt['has_label']).sum():,}, of which feature year "
      f"{config.LABEL_YEAR_END} (no {config.LABEL_YEAR_END + 1} data): {(~mt['has_label'] & (mt['label_year'] == config.LABEL_YEAR_END)).sum()}")
print("Feature columns:", len(L.FEATURE_COLUMNS))
print(L.FEATURE_COLUMNS)

Feature-year rows 2015-2025: 925
  with a t+1 primary label:  749
  without (no next-year row or next-year Z' blank): 176, of which feature year 2025 (no 2026 data): 80
Feature columns: 19
['current_ratio', 'quick_ratio', 'cash_ratio', 'debt_to_equity', 'debt_ratio', 'interest_coverage', 'gross_margin', 'operating_margin', 'net_margin', 'roa', 'roe', 'asset_turnover', 'revenue_growth', 'net_income_growth', 'log_total_assets', 'negative_equity', 'no_interest_expense', 'coverage_uses_proxy', 'low_revenue']


## Step 5: Class balance by feature year (before modelling)

Counts of the t+1 label for each feature year, for all rows and excluding the very small companies flagged `low_revenue`, plus the robustness label and how many rows would change if the grey zone were counted as distressed.

In [7]:
lab = mt[mt["has_label"]].copy()
lab["grey_t1"] = shifted.loc[lab.index, "z_prime_t1"].between(L.DISTRESS_CUTOFF, L.SAFE_CUTOFF, inclusive="left")
balance = lab.groupby("label_year").agg(
    rows=("distressed_t1", "size"),
    distressed_t1=("distressed_t1", "sum"),
    share_distressed=("distressed_t1", "mean"),
)
nolow = lab[~lab["low_revenue"]].groupby("label_year")["distressed_t1"].agg(["size", "sum", "mean"])
balance["rows_excl_low_revenue"] = nolow["size"]
balance["distressed_excl_low_revenue"] = nolow["sum"]
balance["share_excl_low_revenue"] = nolow["mean"]
balance["robust_t1_positive"] = lab.groupby("label_year")["robust_label_t1"].sum()
balance["robust_t1_known"] = lab.groupby("label_year")["robust_label_t1"].count()
balance["grey_t1_would_flip"] = lab.groupby("label_year")["grey_t1"].sum()
balance.loc["All"] = [balance["rows"].sum(), balance["distressed_t1"].sum(), lab["distressed_t1"].mean(),
                      nolow["size"].sum(), nolow["sum"].sum(), lab.loc[~lab["low_revenue"], "distressed_t1"].mean(),
                      balance["robust_t1_positive"].sum(), balance["robust_t1_known"].sum(), balance["grey_t1_would_flip"].sum()]
balance.index.name = "feature_year"
balance.to_csv(config.PROCESSED_DIR / "class_balance.csv")
balance

,rows,distressed_t1,share_distressed,rows_excl_low_revenue,distressed_excl_low_revenue,share_excl_low_revenue,robust_t1_positive,robust_t1_known,grey_t1_would_flip
feature_year,,,,,,,,,
2015,70,32.00,0.46,65,27.00,0.42,2.00,70,20
2016,71,34.00,0.48,67,30.00,0.45,3.00,70,19
2017,77,31.00,0.40,69,24.00,0.35,5.00,76,25
2018,77,35.00,0.45,68,27.00,0.40,3.00,76,27
2019,74,29.00,0.39,65,24.00,0.37,1.00,73,33
2020,72,29.00,0.40,63,20.00,0.32,2.00,72,28
2021,82,31.00,0.38,70,20.00,0.29,3.00,82,34
2022,76,29.00,0.38,69,23.00,0.33,4.00,75,31
2023,76,38.00,0.50,69,31.00,0.45,2.00,75,23


## Step 6: How persistent is distress?

Transitions from this year's label to next year's. If most distressed companies stay distressed, predicting t+1 is close to recognising distress at t; the healthy-to-distressed cases are the genuine early warnings.

In [8]:
trans = pd.crosstab(lab["distressed"], lab["distressed_t1"], rownames=["distressed at t"], colnames=["distressed at t+1"], margins=True)
print(f"Healthy at t that became distressed at t+1: {int(((lab['distressed'] == 0) & (lab['distressed_t1'] == 1)).sum())}")
print(f"Distressed at t that recovered at t+1:      {int(((lab['distressed'] == 1) & (lab['distressed_t1'] == 0)).sum())}")
trans

Healthy at t that became distressed at t+1: 46
Distressed at t that recovered at t+1:      36


distressed at t+1,0.00,1.00,All
distressed at t,,,
0.00,393,46,439
1.00,36,269,305
All,429,315,744


## Step 7: The four case companies

In [9]:
case = zs[zs["cik"].isin(config.CASE_COMPANIES)].copy()
case["company"] = case["cik"].map(config.CASE_COMPANIES)
print("Z' by label year:")
display(case.pivot_table(index="label_year", columns="company", values="z_prime"))
print("Zone by label year:")
display(case.pivot_table(index="label_year", columns="company", values="zone", aggfunc="first"))

Z' by label year:


company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2014,0.11,2.19,2.59,4.68
2015,-1.03,2.06,2.52,2.90
2016,-0.74,1.88,2.93,3.31
2017,0.14,1.86,3.76,2.89
2018,0.75,2.11,4.06,1.73
2019,1.08,1.96,3.37,1.50
2020,2.21,1.88,2.52,0.86
2021,3.02,1.85,2.66,0.37
2022,2.29,1.33,1.96,0.56


Zone by label year:


company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2014,distress,grey,grey,safe
2015,distress,grey,grey,safe
2016,distress,grey,safe,safe
2017,distress,grey,safe,grey
2018,distress,grey,safe,grey
2019,distress,grey,safe,grey
2020,grey,grey,grey,distress
2021,safe,grey,grey,distress
2022,grey,grey,grey,distress


In [10]:
# Components for each case company, to see which term drives the score.
for company in config.CASE_COMPANIES.values():
    print(f"=== {company}")
    display(case[case["company"] == company].set_index("label_year")[["X1", "X2", "X3", "X4", "X5", "z_prime", "zone", "robust_label"]])

=== Nvidia


,X1,X2,X3,X4,X5,z_prime,zone,robust_label
label_year,,,,,,,,
2014,0.67,0.55,0.11,1.59,0.65,2.59,grey,0.00
2015,0.50,0.59,0.10,1.59,0.68,2.52,grey,0.00
2016,0.69,0.62,0.20,1.42,0.70,2.93,safe,0.00
2017,0.72,0.78,0.29,1.98,0.86,3.76,safe,0.00
2018,0.69,0.95,0.29,2.37,0.88,4.06,safe,0.00
2019,0.69,0.86,0.16,2.39,0.63,3.37,safe,0.00
2020,0.42,0.66,0.16,1.42,0.58,2.52,grey,0.00
2021,0.55,0.37,0.23,1.51,0.61,2.66,grey,0.00
2022,0.40,0.25,0.10,1.16,0.65,1.96,grey,0.00


=== AMD


,X1,X2,X3,X4,X5,z_prime,zone,robust_label
label_year,,,,,,,,
2014,0.34,-1.76,-0.04,0.05,1.46,0.11,distress,0.00
2015,0.29,-2.35,-0.15,-0.12,1.28,-1.03,distress,0.00
2016,0.36,-2.35,-0.11,0.14,1.29,-0.74,distress,0.00
2017,0.32,-2.19,0.06,0.21,1.51,0.14,distress,0.00
2018,0.34,-1.63,0.10,0.38,1.42,0.75,distress,0.00
2019,0.37,-1.18,0.10,0.88,1.12,1.08,distress,0.00
2020,0.42,-0.51,0.15,1.87,1.09,2.21,grey,0.00
2021,0.35,-0.12,0.29,1.52,1.32,3.02,safe,0.00
2022,0.13,-0.00,0.02,4.27,0.35,2.29,grey,0.00


=== Intel


,X1,X2,X3,X4,X5,z_prime,zone,robust_label
label_year,,,,,,,,
2014,0.13,0.36,0.17,1.59,0.61,2.19,grey,0.00
2015,0.24,0.36,0.14,1.49,0.54,2.06,grey,0.00
2016,0.13,0.36,0.11,1.43,0.52,1.88,grey,0.00
2017,0.10,0.34,0.15,1.29,0.51,1.86,grey,0.00
2018,0.10,0.39,0.18,1.41,0.55,2.11,grey,0.00
2019,0.07,0.39,0.16,1.32,0.53,1.96,grey,0.00
2020,0.15,0.37,0.15,1.12,0.51,1.88,grey,0.00
2021,0.18,0.41,0.12,1.31,0.47,1.85,grey,0.00
2022,0.10,0.39,0.01,1.29,0.35,1.33,grey,0.00


=== Wolfspeed


,X1,X2,X3,X4,X5,z_prime,zone,robust_label
label_year,,,,,,,,
2014,0.44,0.24,0.04,8.44,0.49,4.68,safe,0.00
2015,0.36,0.06,-0.02,5.05,0.55,2.90,safe,0.00
2016,0.34,-0.00,-0.00,5.95,0.58,3.31,safe,0.00
2017,0.34,-0.08,-0.01,5.20,0.56,2.89,grey,0.00
2018,0.24,-0.18,-0.12,3.66,0.57,1.73,grey,0.00
2019,0.41,-0.30,-0.01,2.62,0.38,1.50,grey,0.00
2020,0.40,-0.32,-0.06,1.82,0.28,0.86,distress,0.00
2021,0.30,-0.45,-0.09,1.59,0.15,0.37,distress,0.00
2022,0.35,-0.45,-0.06,1.65,0.19,0.56,distress,0.00


In [11]:
# The modelling rows of the case companies: label of the following year.
cm = mt[mt["cik"].isin(config.CASE_COMPANIES)].assign(company=lambda d: d["cik"].map(config.CASE_COMPANIES))
cm.pivot_table(index="label_year", columns="company", values="distressed_t1")

company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2015,1.00,0.00,0.00,0.00
2016,1.00,0.00,0.00,0.00
2017,1.00,0.00,0.00,0.00
2018,1.00,0.00,0.00,0.00
2019,0.00,0.00,0.00,1.00
2020,0.00,0.00,0.00,1.00
2021,0.00,0.00,0.00,1.00
2022,0.00,1.00,0.00,1.00
2023,0.00,1.00,0.00,1.00


## Step 8: Save

In [12]:
zs.to_csv(config.PROCESSED_DIR / "zscores.csv", index=False)
out = mt[L.ID_COLUMNS + L.FEATURE_COLUMNS + L.TARGET_COLUMNS + ["has_label"]]
out.to_csv(config.PROCESSED_DIR / "modelling_table.csv", index=False)
print(f"Saved zscores.csv: {len(zs):,} rows")
print(f"Saved modelling_table.csv: {len(out):,} rows ({out['has_label'].sum():,} labelled), "
      f"{len(L.FEATURE_COLUMNS)} features; Z' and X1-X5 not included")

Saved zscores.csv: 1,028 rows

Saved modelling_table.csv: 925 rows (749 labelled), 19 features; Z' and X1-X5 not included
